# Lesson 3.1 — Business iQ and Analytics Overview

## Learning Objectives
- Understand what Business iQ (Analytics) adds beyond standard APM metrics
- Describe the Events Service and its role in Analytics
- Identify the key analytics tables available in AppDynamics
- Navigate to the Analytics Query UI in the Controller

**Reference**: [Analytics Overview](https://help.splunk.com/en/appdynamics-saas/analytics/26.2.0/analytics/overview-of-analytics) | [Using Analytics Data](https://help.splunk.com/en/appdynamics-saas/analytics/26.2.0/analytics/using-analytics-data)

## 1. What is Business iQ?

Standard AppDynamics APM gives you **aggregated metrics**: average response time, calls per minute, error rate.

**Business iQ** (the Analytics module) gives you the **raw events** behind those aggregates:

| Standard Metrics | Business iQ Analytics |
|-----------------|------------------------|
| Avg response time: 320ms | 1,240 individual transactions, each with exact time, user, device, region |
| Error rate: 2.3% | The specific 28 errored transactions with error messages, stack traces, user IDs |
| Calls/min: 45 | Which users? Which regions? Which products? What browser? |

Business iQ answers questions like:
- *Which users experienced errors in the last hour?*
- *What's the average response time for mobile users vs. desktop users?*
- *Which product SKUs are in the slowest checkout transactions?*
- *How many page views came from Chrome vs. Firefox?*

These business-level insights require **ADQL** — AppDynamics Query Language.

## 2. The Events Service

Business iQ data lives in the **Events Service** — a high-volume, document-oriented store separate from the Metrics Store.

### Data flow:
```
App Server Agent
    │
    ├──► Metrics Store (aggregated, every 1 minute)
    │    └─ "Avg RT was 320ms from 14:00-14:01"
    │
    └──► Events Service (individual events, real-time)
         └─ {transactionName: /checkout, responseTime: 287ms, userId: 1042, ...}
         └─ {transactionName: /checkout, responseTime: 4312ms, userId: 5581, ...} ← slow!
         └─ ... (all 1,240 transactions individually stored)
```

### What requires an Analytics Agent:
- **Business Transaction analytics** — automatic if Analytics Agent is deployed
- **Log analytics** — requires configuring log collection in the Analytics Agent
- **Custom analytics** — you can publish custom events via API

In [ ]:
# Demonstrate what individual transaction records look like (Events Service model)

import random, json
from datetime import datetime, timedelta

random.seed(77)

def make_transaction_record(ts: datetime) -> dict:
    """Simulate an individual transaction record as stored in the Events Service."""
    is_error = random.random() < 0.04
    user_exp = random.choices(
        ["NORMAL", "SLOW", "VERY_SLOW", "ERROR"],
        weights=[85, 9, 4, 2]
    )[0]
    rt = {
        "NORMAL":   random.gauss(280, 60),
        "SLOW":     random.gauss(950, 100),
        "VERY_SLOW": random.gauss(2100, 300),
        "ERROR":    random.gauss(1200, 400)
    }[user_exp]
    
    return {
        "eventTimestamp": int(ts.timestamp() * 1000),
        "transactionName": random.choice(["/checkout", "/search", "/cart/add"]),
        "applicationName": "StoreFront",
        "tierName": "WebFrontend",
        "responseTime": max(50, rt),
        "userExperience": user_exp if not is_error else "ERROR",
        "errorOccurred": is_error,
        "userId": random.randint(1000, 9999),
        "ipAddress": f"10.{random.randint(1,254)}.{random.randint(1,254)}.{random.randint(1,254)}",
        "browser": random.choice(["Chrome", "Firefox", "Safari", "Edge"]),
        "country": random.choice(["United States", "Germany", "United Kingdom", "Japan", "Brazil"]),
        "deviceType": random.choice(["Desktop", "Mobile", "Tablet"])
    }

# Generate 10 sample records
base_ts = datetime.now() - timedelta(minutes=5)
sample_records = [make_transaction_record(base_ts + timedelta(seconds=i*3)) for i in range(10)]

print("Sample Events Service records (what ADQL queries against):")
print(json.dumps(sample_records[0], indent=2))
print(f"\n...and {len(sample_records)-1} more records. In production, millions per hour.")

## 3. Analytics Tables in AppDynamics

ADQL queries run against specific **tables** in the Events Service. Here are the key tables:

| Table Name | Contains | Typical Use |
|------------|---------|-------------|
| `transactions` | APM Business Transaction events | Response time analysis, error diagnosis, user experience |
| `browser_records` | Browser RUM page load events | Page performance, AJAX timing, geographic analysis |
| `mobile_snapshots` | Mobile APM crash and request data | Mobile performance, crash rates |
| `application_logs` | Application log events | Error investigation, log-to-transaction correlation |
| `custom_events` | User-defined events published via API | Business KPI tracking, custom analytics |

### Key fields in `transactions`:

| Field | Type | Description |
|-------|------|-------------|
| `eventTimestamp` | Long | Unix epoch in milliseconds |
| `transactionName` | String | The BT name (e.g., `/checkout`) |
| `applicationName` | String | Application name |
| `tierName` | String | Tier where BT started |
| `responseTime` | Long | Response time in milliseconds |
| `userExperience` | String | NORMAL, SLOW, VERY_SLOW, STALL, ERROR |
| `errorOccurred` | Boolean | True if an error occurred |
| `userId` | String | User ID (if configured) |
| `httpRequestMethod` | String | GET, POST, etc. |
| `httpStatus` | Integer | HTTP response status code |

## 4. Accessing the Analytics Query UI

### Steps to open the Analytics Query UI:
1. Log in to the AppDynamics Controller UI
2. In the left navigation, click **Analytics**
   - (May be labeled **Business iQ** in older versions)
3. Click **Searches**
4. Click **Add** (or the + button) to create a new search
5. Choose **ADQL Query** as the query type
6. Type your ADQL query in the editor
7. Click **Run** (or press Ctrl+Enter)

### Important note on the Analytics Agent:
- Business Transaction analytics requires the **Analytics Agent** to be deployed alongside the App Server Agent
- Without the Analytics Agent, the `transactions` table will be empty
- Check **Settings → Analytics → Analytics Agents** in the Controller UI to verify connectivity

### Time range:
- Analytics queries are time-bounded by the **time range selector** in the UI (top right)
- Default: last 30 minutes
- Maximum window: typically 30 days (depends on retention configuration)

In [ ]:
# The Analytics REST API endpoint
# AppDynamics provides an Analytics REST API for running ADQL queries programmatically

import os, requests

CONTROLLER_URL    = os.getenv("APPD_CONTROLLER_URL",  "https://your-account.saas.appdynamics.com")
ANALYTICS_API_URL = os.getenv("APPD_ANALYTICS_URL",   "https://analytics.api.appdynamics.com")
ACCOUNT_NAME      = os.getenv("APPD_ACCOUNT",          "your-account")
API_KEY           = os.getenv("APPD_ANALYTICS_API_KEY", "your-api-key")  # Analytics-specific API key
GLOBAL_ACCOUNT    = os.getenv("APPD_GLOBAL_ACCOUNT",    "your-global-account-name")

def run_adql(query: str, start_ms: int = None, end_ms: int = None, limit: int = 100) -> dict:
    """
    Execute an ADQL query against the AppDynamics Analytics REST API.
    
    Authentication uses X-Events-API-AccountName + X-Events-API-Key headers.
    This is DIFFERENT from the Controller REST API auth.
    """
    import time
    end_ms   = end_ms   or int(time.time() * 1000)
    start_ms = start_ms or (end_ms - 30 * 60 * 1000)  # 30 minutes ago
    
    url = f"{ANALYTICS_API_URL}/events/query"
    headers = {
        "Content-Type": "application/vnd.appd.events+json;v=2",
        "Accept": "application/vnd.appd.events+json;v=2",
        "X-Events-API-AccountName": GLOBAL_ACCOUNT,
        "X-Events-API-Key": API_KEY,
    }
    payload = {
        "query": query,
        "startTime": start_ms,
        "endTime": end_ms,
        "limit": limit
    }
    resp = requests.post(url, json=payload, headers=headers, timeout=30)
    resp.raise_for_status()
    return resp.json()

print("run_adql() helper function defined.")
print()
print("NOTE: Analytics API requires different credentials than the Controller REST API:")
print("  - X-Events-API-AccountName: your GLOBAL account name (from Account Management Portal)")
print("  - X-Events-API-Key: your Analytics API key")
print("  Find these in: Controller UI → Settings → Analytics → Analytics API Keys")

In [ ]:
# Test query: count all transactions in the last 30 minutes

test_query = "SELECT count(*) FROM transactions"

try:
    result = run_adql(test_query)
    print(f"Query: {test_query}")
    print(f"Result: {result}")
except Exception as e:
    print(f"[Not connected] Expected result format:")
    print()
    print("Query:", test_query)
    print("Result:")
    sample_result = {
        "results": [{"count(*)": 14372}],
        "fields": ["count(*)"],
        "moreData": False
    }
    import json
    print(json.dumps(sample_result, indent=2))

## Summary

- **Business iQ** provides individual event-level data beyond aggregated APM metrics
- Data lives in the **Events Service** — a separate document store, not the Metrics Store
- Key tables: `transactions`, `browser_records`, `mobile_snapshots`, `application_logs`
- ADQL queries the Events Service — the Analytics REST API uses different auth headers than the Controller REST API
- Access the query UI via **Analytics → Searches** in the Controller

---

**Next**: [Lesson 3.2 — ADQL Fundamentals](lesson-3.2-adql-fundamentals.ipynb)